<a href="https://colab.research.google.com/github/habtew/nlp/blob/enamtransformers/transformers_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import math
# hello this is testing if it works or not

In [ ]:
class InputEmbeddings(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, seq_len: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        # create matrix len of (seq_len, d_model)
        pe = torch.zeros(seq_len, d_model)

        # create vector of shape(seq_len, 1)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        # apply the sin to even position
        pe[:, 0::2] = torch.sin(position * div_term)
        # apply the cos to odd position
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0) # (1, seq_len, d_model)

        self.register_buffer('pe', pe)

    # def forward(self, x):
    #   x = x + (self.pe[:, :x.shape(1), :]).requires_grad_(False)
    #   return self.dropout(x)
    def forward(self, x):
        # Change x.shape(1) to x.shape[1] to access the second dimension using indexing
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        return self.dropout(x)

In [ ]:
class LayerNormalization(nn.Module):
  def __init__(self, ep=1e-5) -> None:
    super().__init__()
    self.ep = ep
    self.alpha = nn.Parameter(torch.ones(1)) # multplied
    self.bias = nn.Parameter(torch.zeros(1)) # added

  def forward(self, x):
    mean = x.mean(dim = -1, keepdim=True)
    std = x.std(dim = -1, keepdim=True, unbiased=False)
    return self.alpha * (x - mean) / (std + self.ep) + self.bias

In [ ]:
# feedforward layer
class FeedForwardBloc(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff) #w1 and b1
        self.dropout = nn.Dropout(dropout) #
        self.linear_2 = nn.Linear(d_ff, d_model) #w2 and b2

    def forward(self, x):
        # (batch, seq_len, d_model) --> (batch, seq_len, d_ff) --> (batch, seq_len, d_model)
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

In [ ]:
# multi head attention block
class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model: int, h: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.h = h
        assert d_model % h == 0, "d_model is not divisible by h"
        self.d_k = d_model // h
        self.w_q = nn.Linear(d_model, d_model) #wq
        self.w_k = nn.Linear(d_model, d_model) #wk
        self.w_v = nn.Linear(d_model, d_model) #wv

        self.w_o = nn.Linear(d_model, d_model) #wo
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.size(-1)
        # attention_scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e9)
        attention_scores = attention_scores.softmax(dim=-1)
        if dropout is not None:
            attention_scores = dropout(attention_scores)
        return (attention_scores @ value), attention_scores

    def forward(self, q, k, v, mask):
        # batch_size = q.size(0)
        query = self.w_q(q) #(batch, seq_len, d_model)
        key = self.w_k(k) #(batch, seq_len, d_model)
        # Changed to using w_v to get the value instead of using w_k
        value = self.w_v(v) #(batch, seq_len, d_model)

        # (batch, seq_len, d_model) --> (batch, seq_len, h, d_k ) --> (batch, h, seq_len, d_k)
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        # Use key.shape instead of query.shape to reshape the key
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)
        # Use value.shape instead of query.shape to reshape the value
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)

        # (batch, h, seq_len, d_k) --> (batch, seq_len, h, d_k) ---> (batch, seq_len, d_model)
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)
        # (batch, seq_len, d_model)
        return self.w_o(x)

In [ ]:
class ResidualConnection(nn.Module):
    def __init__(self, dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = LayerNormalization(ep=1e-6)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, self_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBloc, dropout: float):
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block

        self.residual_connections = nn.ModuleList([ResidualConnection(dropout) for _ in range(2)])

    def forward(self, x, src_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, src_mask))
        x = self.residual_connections[1](x, self.feed_forward_block)
        return x

In [ ]:
class Encoder(nn.Module):
    def __init__(self, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(ep=1e-6)
    def forward(self, x, mask) :
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, self_attention_block: MultiHeadAttentionBlock, cross_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBloc, dropout: float):
        super().__init__()
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_block = feed_forward_block

        self.residual_connections = nn.ModuleList([ResidualConnection(dropout) for _ in range(3)])

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, tgt_mask))
        x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, src_mask))
        x = self.residual_connections[2](x, self.feed_forward_block)
        return x

In [ ]:
class Decoder(nn.Module):
    def __init__(self, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(ep=1e-6)

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)

        # Apply LayerNormalization to the output before returning
        return self.norm(x) # This line was changed from 'return self.norm'

In [ ]:
class ProjectLayer(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # (batch, seq_len, d_model) ---> (batch, seq_len, vocab_size)
        return torch.log_softmax(self.proj(x), dim = -1)

In [ ]:
class Transformer(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, src_embed: InputEmbeddings, tgt_embed: InputEmbeddings, src_pos: PositionalEncoding, tgt_pos: PositionalEncoding, projection_layer: ProjectLayer):
      super().__init__()
      self.encoder = encoder
      self.decoder = decoder
      self.src_embed = src_embed
      self.tgt_embed = tgt_embed
      self.src_pos = src_pos
      self.tgt_pos = tgt_pos
      self.projection_layer = projection_layer


    def encode(self, src, src_mask):
      src = self.src_embed(src)
      src = self.src_pos(src)
      return self.encoder(src, src_mask)

    # Renamed decoder to decode to match the call in train_model
    def decode(self, encoder_output, src_mask, tgt, tgt_mask):
      tgt = self.tgt_embed(tgt)
      tgt = self.tgt_pos(tgt)
      return self.decoder(tgt, encoder_output, src_mask, tgt_mask)

    def project(self, x):
      return self.projection_layer(x)

In [ ]:
def build_transformer(src_vocab_size: int, tgt_vocab_size: int, src_seq_len: int, tgt_seq_len: int, d_model: int = 512, N: int = 6, h: int = 8, d_ff: int = 2048, dropout: float = 0.1) -> Transformer:
  # create embedding layers
  src_embed = InputEmbeddings(d_model, src_vocab_size)
  tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)

  # create positional encode layers
  src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
  tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout)

  # create the encoder blocks
  encoder_blocks = []
  for _ in range(N):
    self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
    feed_forward_block = FeedForwardBloc(d_model, d_ff, dropout)
    encode_block = EncoderBlock(self_attention_block, feed_forward_block, dropout)
    encoder_blocks.append(encode_block)


  # create the decoder blocks
  decoder_blocks = []
  for _ in range(N):
    decoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
    decoder_cross_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
    feed_forward_block = FeedForwardBloc(d_model, d_ff, dropout)
    decoder_block = DecoderBlock(decoder_self_attention_block, decoder_cross_attention_block, feed_forward_block, dropout)
    decoder_blocks.append(decoder_block)

  # create encoder and decoder
  encoder = Encoder(nn.ModuleList(encoder_blocks))
  decoder = Decoder(nn.ModuleList(decoder_blocks))

  # create the projection layer
  projection_layer = ProjectLayer(d_model, tgt_vocab_size)

  # build the transformer
  transformer = Transformer(encoder, decoder, src_embed, tgt_embed, src_pos, tgt_pos, projection_layer)
  # initialize paramaters to make training faster
  for p in transformer.parameters():
    if p.dim() > 1:
      nn.init.xavier_uniform_(p)

  return transformer

In [ ]:
!pip install datasets
!pip install tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_syst

In [ ]:
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
from torch.utils.data import DataLoader, Dataset

# from pathlib import Path

# def get_all_sentences(ds, lang):
#   for item in ds:
#     yield item['translation'][lang]

# def get_or_build(config, ds, lang):
#   tokenizer_path = Path(config['tokenizer_file'].format(lang))
#   if not Path.exists(tokenizer_path):
#     tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
#     tokenizer.pre_tokenizer = Whitespace()
#     trainer = WordLevelTrainer(special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"], min_frequency=2)
#     tokenizer.train_from_iterator(get_all_sentences(ds, lang), trainer=trainer)
#     tokenizer.save(str(tokenizer_path))
#   else:
#     tokenizer = Tokenizer.from_file(str(tokenizer_path))
#   return tokenizer

# def get_ds(config):
#   # ds_raw = load_dataset("opus_books", f"{config['lang_src']}-{config['lang_tgt']}", split='train')
#   ds_raw = load_dataset("habtew/english-amharic-translation", split='train')

#   # build tokenizer
#   tokenizer_src = get_or_build(config, ds_raw, config['lang_src'])
#   tokenizer_tgt = get_or_build(config, ds_raw, config['lang_tgt'])

#   # validation data
#   ds_valid = load_dataset("habtew/english-amharic-translation",  split='validation')








In [ ]:
def get_config():
  return {
      "batch_size": 8,
      "num_epochs": 1,
      "lr": 10**-4,
      "seq_len": 370,
      "d_model": 512,
      "lang_src": "en",
      "lang_tgt": "am",
      "model_folder": "weights",
      "model_basename": "tmodel_",
      # "preload": None,
      "preload": "00",
      "tokenizer_file": "tokenizer_{0}.json",
      "experiment_name": "runs/tmodel"
  }

def get_weights_file_path(config, epoch: str):
  model_folder = config['model_folder']
  model_basename = config['model_basename']
  model_filename = f"{model_basename}{epoch}.pt"
  return str(Path('.') / model_folder / model_filename)

In [ ]:
# dataset
class BilingualDataset(Dataset):
  def __init__(self, ds, tokenizer_src, tokenizer_tgt, src_lang, tgt_lang, seq_len):
    super().__init__()
    self.ds = ds
    self.seq_len = seq_len
    self.tokenizer_src = tokenizer_src
    self.tokenizer_tgt = tokenizer_tgt
    self.src_lang = src_lang
    self.tgt_lang = tgt_lang

    self.sos_token = torch.tensor([tokenizer_src.get_vocab()['[SOS]']], dtype=torch.int64)
    self.eos_token = torch.tensor([tokenizer_src.get_vocab()['[EOS]']], dtype=torch.int64)
    self.pad_token = torch.tensor([tokenizer_src.get_vocab()['[PAD]']], dtype=torch.int64)

  def __len__(self):
    return len(self.ds)

  def __getitem__(self, index):
    src_target_pair = self.ds[index]
    src_text = src_target_pair['translation'][self.src_lang]
    tgt_text = src_target_pair['translation'][self.tgt_lang]

    # transform the text into tokens
    enc_input_tokens = self.tokenizer_src.encode(src_text).ids
    dec_input_tokens = self.tokenizer_tgt.encode(tgt_text).ids

    # add sos and eos tokens
    enc_num_padding_tokens = self.seq_len - len(enc_input_tokens) - 2
    dec_num_padding_tokens = self.seq_len - len(dec_input_tokens) - 1

    if enc_num_padding_tokens < 0 or dec_num_padding_tokens < 0:
      raise ValueError("Sentence is too long")

    # add sos and eos to src text
    encoder_input = torch.cat(
        [
            self.sos_token,
            torch.tensor(enc_input_tokens, dtype=torch.int64),
            self.eos_token,
            torch.tensor([self.pad_token] * enc_num_padding_tokens, dtype=torch.int64),
        ]

        )
    #  decoder input

    decoder_input = torch.cat(
        [
            self.sos_token,
            torch.tensor(dec_input_tokens, dtype=torch.int64),
            torch.tensor([self.pad_token] * dec_num_padding_tokens, dtype=torch.int64),
        ]
    )

    # target
    label = torch.cat(
        [
            torch.tensor(dec_input_tokens, dtype=torch.int64),
            self.eos_token,
            torch.tensor([self.pad_token] * dec_num_padding_tokens, dtype=torch.int64),
        ]
    )

    assert encoder_input.size(0) == self.seq_len
    assert decoder_input.size(0) == self.seq_len
    assert label.size(0) == self.seq_len

    return {
        "encoder_input": encoder_input,
        "decoder_input": decoder_input,
        "encoder_mask": (encoder_input != self.pad_token).unsqueeze(0).unsqueeze(0).int(),
        "decoder_mask": (decoder_input != self.pad_token).unsqueeze(0).int() & causal_mask(decoder_input.size(0)),
        "label": label,
        "src_text": src_text,
        "tgt_text": tgt_text,
    }

def causal_mask(size):
  mask = torch.triu(torch.ones((1, size, size)), diagonal=1).type(torch.int)
  return mask == 0

In [ ]:
!pip install torchmetrics==0.11.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 26.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [ ]:
import torchmetrics
import os
def greedy_decode(model, source, source_mask, tokenizer_src, tokenizer_tgt, max_len, device):
    sos_idx = tokenizer_tgt.token_to_id('[SOS]')
    eos_idx = tokenizer_tgt.token_to_id('[EOS]')

    # Precompute the encoder output and reuse it for every step
    encoder_output = model.encode(source, source_mask)
    # Initialize the decoder input with the sos token
    decoder_input = torch.empty(1, 1).fill_(sos_idx).type_as(source).to(device)
    while True:
        if decoder_input.size(1) == max_len:
            break

        # build mask for target
        decoder_mask = causal_mask(decoder_input.size(1)).type_as(source_mask).to(device)

        # calculate output
        out = model.decode(encoder_output, source_mask, decoder_input, decoder_mask)

        # get next token
        prob = model.project(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        decoder_input = torch.cat(
            [decoder_input, torch.empty(1, 1).type_as(source).fill_(next_word.item()).to(device)], dim=1
        )

        if next_word == eos_idx:
            break

    return decoder_input.squeeze(0)


def run_validation(model, validation_ds, tokenizer_src, tokenizer_tgt, max_len, device, print_msg, global_step, writer, num_examples=2):
    model.eval()
    count = 0

    source_texts = []
    expected = []
    predicted = []

    try:
        # get the console window width
        with os.popen('stty size', 'r') as console:
            _, console_width = console.read().split()
            console_width = int(console_width)
    except:
        # If we can't get the console width, use 80 as default
        console_width = 80

    with torch.no_grad():
        for batch in validation_ds:
            count += 1
            encoder_input = batch["encoder_input"].to(device) # (b, seq_len)
            encoder_mask = batch["encoder_mask"].to(device) # (b, 1, 1, seq_len)

            # check that the batch size is 1
            assert encoder_input.size(
                0) == 1, "Batch size must be 1 for validation"

            model_out = greedy_decode(model, encoder_input, encoder_mask, tokenizer_src, tokenizer_tgt, max_len, device)

            source_text = batch["src_text"][0]
            target_text = batch["tgt_text"][0]
            model_out_text = tokenizer_tgt.decode(model_out.detach().cpu().numpy())

            source_texts.append(source_text)
            expected.append(target_text)
            predicted.append(model_out_text)

            # Print the source, target and model output
            print_msg('-'*console_width)
            print_msg(f"{f'SOURCE: ':>12}{source_text}")
            print_msg(f"{f'TARGET: ':>12}{target_text}")
            print_msg(f"{f'PREDICTED: ':>12}{model_out_text}")

            if count == num_examples:
                print_msg('-'*console_width)
                break

    if writer:
        # Evaluate the character error rate
        # Compute the char error rate
        metric = torchmetrics.CharErrorRate()
        cer = metric(predicted, expected)
        writer.add_scalar('validation cer', cer, global_step)
        writer.flush()

        # Compute the word error rate
        metric = torchmetrics.WordErrorRate()
        wer = metric(predicted, expected)
        writer.add_scalar('validation wer', wer, global_step)
        writer.flush()

        # Compute the BLEU metric
        metric = torchmetrics.BLEUScore()
        bleu = metric(predicted, expected)
        writer.add_scalar('validation BLEU', bleu, global_step)
        writer.flush()

In [ ]:
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from torch.utils.data import random_split


def get_all_sentences(ds, lang):
  for item in ds:
    yield item['translation'][lang]

def get_or_build(config, ds, lang):
  tokenizer_path = Path(config['tokenizer_file'].format(lang))
  if not Path.exists(tokenizer_path):
    tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = WordLevelTrainer(special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"], min_frequency=2)
    tokenizer.train_from_iterator(get_all_sentences(ds, lang), trainer=trainer)
    tokenizer.save(str(tokenizer_path))
  else:
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
  return tokenizer

def get_ds(config):
  # ds_raw = load_dataset("opus_books", f"{config['lang_src']}-{config['lang_tgt']}", split='train')
  # ds_raw = load_dataset("habtew/en-am-dataset", split='train')
  # #####
  # ds_raw = load_dataset("habtew/en-am-dataset", split='train')
  # #####
  ds_raw = load_dataset("habtew/english-amharic-translation", split="train")
  # ds_raw = load_dataset(f"{config['datasource']}", f"{config['lang_src']}-{config['lang_tgt']}", split='train')

  # build tokenizer
  tokenizer_src = get_or_build(config, ds_raw, config['lang_src'])
  tokenizer_tgt = get_or_build(config, ds_raw, config['lang_tgt'])


  train_ds_size = int(0.9 * len(ds_raw))  # 20% for training
  val_ds_size = len(ds_raw) - train_ds_size  # Remaining 80% for validation
  train_ds_raw, val_ds_raw = random_split(ds_raw, [train_ds_size, val_ds_size])
  # validation data
  # ds_valid = load_dataset("habtew/english-amharic-translation",  split='validation')

  # train_ds = BilingualDataset(ds_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])
  # valid_ds = BilingualDataset(ds_valid, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])

  train_ds = BilingualDataset(train_ds_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])
  val_ds = BilingualDataset(val_ds_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])


  max_len_src = 0
  max_len_tgt = 0

  for item in ds_raw:
    src_ids = tokenizer_src.encode(item['translation'][config['lang_src']]).ids
    tgt_ids = tokenizer_tgt.encode(item['translation'][config['lang_tgt']]).ids
    max_len_src = max(max_len_src, len(src_ids))
    max_len_tgt = max(max_len_tgt, len(tgt_ids))

  print(f"Max length of source sentence: {max_len_src}")
  print(f"Max length of target sentence: {max_len_tgt}")

  # load training data
  train_dataloader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
  val_dataloader = DataLoader(val_ds, batch_size=1, shuffle=True)

  return train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt

def get_model(config, vocab_src_len, vocab_tgt_len):
  model = build_transformer(vocab_src_len, vocab_tgt_len, config['seq_len'], config['seq_len'], d_model=config['d_model'])
  return model


def train_model(config):
  # define the device
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print("Using device:", device)

  Path(config['model_folder']).mkdir(parents=True, exist_ok=True)
  train_dataloader, valid_dataloader, tokenizer_src, tokenizer_tgt = get_ds(config)
  model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)

  # start tensorboard

  writer = SummaryWriter(config['experiment_name'])

  optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'], eps=1e-9)

  initial_epoch = 0
  global_step = 0
  if config['preload']:
    model_filename = get_weights_file_path(config, config['preload'])
    print(f"Preloading model {model_filename}")
    state = torch.load(model_filename)
    initial_epoch = state['epoch'] + 1
    optimizer.load_state_dict(state['optimizer_state_dict'])
    global_step = state['global_step']

  loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer_src.token_to_id('[PAD]'), label_smoothing=0.1).to(device)

  # training loop
  for epoch in range(initial_epoch, config['num_epochs']):
    model.train()
    batch_iterator = tqdm(train_dataloader, desc=f"Processing Epoch {epoch:02d}")

    for batch in batch_iterator:
      encoder_input = batch['encoder_input'].to(device) # (batch, seq_len)
      decoder_input = batch['decoder_input'].to(device) # (batch, seq_len)
      encoder_mask = batch['encoder_mask'].to(device) # (batch, 1, 1, seq_len)
      decoder_mask = batch['decoder_mask'].to(device) # (batch, 1, seq_len, seq_len)

      # run the tensors through the transformer
      encoder_output = model.encode(encoder_input, encoder_mask) # (batch, seq_len, d_model)
      decoder_output = model.decode(encoder_output, encoder_mask, decoder_input, decoder_mask) # (batch, seq_len, d_model)
      proj_output = model.project(decoder_output) # (batch, seq_len, vocab_size)

      label = batch['label'].to(device) # (batch, seq_len)

      # (Batch, seq_len, tgt_vocab_size) --- > (Batch * seq_len, tgt_vocab_size)
      loss = loss_fn(proj_output.view(-1, tokenizer_tgt.get_vocab_size()), label.view(-1))

      # show loss
      batch_iterator.set_postfix({"loss": f"{loss.item():6.3f}"})
      # log to tensorboar
      writer.add_scalar('train loss', loss.item(), global_step)
      writer.flush()

      # backpropagate
      loss.backward()

      # update the weights of the model
      optimizer.step()
      optimizer.zero_grad()

      global_step += 1
    # run validation at the end of every epoch
    run_validation(model, valid_dataloader, tokenizer_src, tokenizer_tgt, config['seq_len'], device, lambda msg: batch_iterator.write(msg), global_step, writer)
    # save the model
    model_filename = get_weights_file_path(config, f"{epoch:02d}")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'global_step': global_step
    }, model_filename)
    print(f"data loader len datasetsize / batchsize = {len(train_dataloader)}")

if __name__ == "__main__":
  config = get_config()
  train_model(config)

Using device: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/591 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/30.3M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/3.77M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/21568 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/43135 [00:00<?, ? examples/s]

Max length of source sentence: 291
Max length of target sentence: 207


Processing Epoch 00: 100%|██████████| 19411/19411 [2:30:14<00:00,  2.15it/s, loss=5.816]


--------------------------------------------------------------------------------
    SOURCE: soon, the heavy iron bearing chariots were more of a liability than an asset.
    TARGET: ብዙ የብረት መሳሪያ የተገጠመላቸው ሰረገላዎች ከጥቅማቸው ይልቅ ጉዳታቸው አመዘነ።
 PREDICTED: ።
--------------------------------------------------------------------------------
    SOURCE: god promises to eliminate suffering and injustice, "making all things new."
    TARGET: አምላክ መከራንና የፍትህ መዛባትን በማስወገድ "ሁሉንም ነገር አዲስ ለማድረግ" ቃል ገብቷል።
 PREDICTED: አምላክ ፣ አምላክ እንዲሁም " " እንደሆነ ያሳያል ።
--------------------------------------------------------------------------------
data loader len datasetsize / batchsize = 19411


In [ ]:
import torch
from pathlib import Path
from tokenizers import Tokenizer
# from model import build_transformer  # Ensure this is the same model used in training
# from dataset import BilingualDataset

def load_model(config, device):
    # Get the vocabulary sizes from the tokenizers
    tokenizer_src = Tokenizer.from_file(config['tokenizer_file'].format(config['lang_src']))
    tokenizer_tgt = Tokenizer.from_file(config['tokenizer_file'].format(config['lang_tgt']))
    src_vocab_size = tokenizer_src.get_vocab_size()
    tgt_vocab_size = tokenizer_tgt.get_vocab_size()

    # Build the transformer with the correct vocabulary sizes
    model = build_transformer(
        src_vocab_size,
        tgt_vocab_size,
        config['seq_len'],
        config['seq_len'],
        d_model=config['d_model']
    ).to(device)

    model_path = get_weights_file_path(config, config['preload'])
    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state['model_state_dict'])
    model.eval()
    return model


def translate_sentence(sentence, model, tokenizer_src, tokenizer_tgt, config, device):
    src_ids = tokenizer_src.encode(sentence).ids[:config['seq_len']]
    src_tensor = torch.tensor([src_ids], dtype=torch.long).to(device)

    encoder_mask = (src_tensor != tokenizer_src.token_to_id('[PAD]')).unsqueeze(1).unsqueeze(2)
    encoder_output = model.encode(src_tensor, encoder_mask)

    tgt_ids = [tokenizer_tgt.token_to_id('[SOS]')]

    for _ in range(config['seq_len']):
        tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long).to(device)
        decoder_mask = torch.tril(torch.ones((1, len(tgt_ids), len(tgt_ids)), dtype=torch.bool)).to(device)

        decoder_output = model.decode(encoder_output, encoder_mask, tgt_tensor, decoder_mask)
        proj_output = model.project(decoder_output[:, -1])

        next_token = torch.argmax(proj_output, dim=-1).item()
        if next_token == tokenizer_tgt.token_to_id('[EOS]'):
            break
        tgt_ids.append(next_token)

    translated_sentence = tokenizer_tgt.decode(tgt_ids[1:])
    return translated_sentence

def main():
    config = get_config()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = load_model(config, device)
    tokenizer_src = Tokenizer.from_file(config['tokenizer_file'].format(config['lang_src']))
    tokenizer_tgt = Tokenizer.from_file(config['tokenizer_file'].format(config['lang_tgt']))
    sentence = "this is me"
    translation = translate_sentence(sentence, model, tokenizer_src, tokenizer_tgt, config, device)
    print(f"english: {sentence}\namharic:{translation}")

    # while True:
    #     sentence = input("Enter an English sentence: ")
    #     if sentence.lower() == 'exit':
    #         break
        # translation = translate_sentence(sentence, model, tokenizer_src, tokenizer_tgt, config, device)
    #     print("Amharic Translation:", translation)

if __name__ == "__main__":
    main()

<ipython-input-39-f4dcd231ba42>:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(model_path, map_location=device)


english: this is me
amharic:ይህ ነው ፣


In [ ]:
ds_raw = load_dataset("habtew/en-am-dataset", split='train')
# ds_raw = load_dataset(f"{config['datasource']}", f"{config['lang_src']}-{config['lang_tgt']}", split='train')
print(ds_raw)
# build tokenizer
tokenizer_src = get_or_build(config, ds_raw, config['lang_src'])
tokenizer_tgt = get_or_build(config, ds_raw, config['lang_tgt'])


train_ds_size = int(0.9 * len(ds_raw))  # 20% for training
print(train_ds_size)
val_ds_size = len(ds_raw) - train_ds_size  # Remaining 80% for validation
print(val_ds_size)
train_ds_raw, val_ds_raw = random_split(ds_raw, [train_ds_size, val_ds_size])
# validation data
# ds_valid = load_dataset("habtew/english-amharic-translation",  split='validation')

# train_ds = BilingualDataset(ds_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])
# valid_ds = BilingualDataset(ds_valid, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])

train_ds = BilingualDataset(train_ds_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])
val_ds = BilingualDataset(val_ds_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])
print(train_ds_raw)

README.md:   0%|          | 0.00/569 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/4.35k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.75k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/2.18k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/21 [00:00<?, ? examples/s]

Dataset({
    features: ['translation'],
    num_rows: 84
})
75
9


In [ ]:
from datasets import load_dataset

# dataset = load_dataset("habtew/english-amharic-translation")
dataset = load_dataset("habtew/en-am-dataset")
# print(dataset['train'][0]['translation']['en'])
ds_ds = dataset['train']
maxi = 0
maxi_ = 0
maxii = ''
maxiii = ''
print(dataset)
for item in ds_ds:
    src_ids = len(item['translation']['en'])
    tgt_ids = len(item['translation']['am'])
    if maxi == 1343:
      maxii = item['translation']['en']

    if maxi_ == 895:
      maxiii = item['translation']['am']

    maxi = max(maxi, src_ids)
    maxi_ = max(maxi_, tgt_ids)

print(maxi, maxi_)
print(maxii)
print(maxiii)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/569 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/4.35k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.75k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/2.18k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/21 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 84
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 11
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 21
    })
})
34 24




In [ ]:
import os
import sys
from pathlib import Path

def latest_weights_file_path(config):
  """
  Returns the path to the latest weights file based on the model folder and basename.
  """
  model_folder = Path(config['model_folder'])
  model_basename = config['model_basename']
  # Find all weight files in the model folder
  weight_files = list(model_folder.glob(f"{model_basename}*.pt"))
  # Sort the files by modification time (latest first)
  weight_files.sort(key=os.path.getmtime, reverse=True)
  # If there are any weight files, return the path to the latest one
  if weight_files:
      return str(weight_files[0])
  else:
      raise FileNotFoundError("No weight files found in the model folder.")